In [1]:
import pandas as pd
import os
import numpy as np

In [4]:
# Function to calculate harvest intensity
def calculate_harvest_intensity(harvest_df, physical_df):
    """
    Calculates the harvest intensity by dividing each column in the harvest DataFrame 
    by the corresponding column in the physical area DataFrame.

    Args:
    harvest_df (DataFrame): A DataFrame containing harvest data.
    physical_df (DataFrame): A DataFrame containing physical area data.

    Returns:
    DataFrame: A DataFrame with calculated harvest intensity.
    """
    intensity_df = harvest_df.copy()
    for col in intensity_df.columns[9:-6]:  # Iterate over the specific columns
        intensity_df[col] = harvest_df[col] / physical_df[col]
    return intensity_df

# Function to calculate point-in-time production
def calculate_pit_production(production_df, intensity_df):
    """
    Calculates the point-in-time production by dividing each column in the production DataFrame 
    by the corresponding column in the intensity DataFrame.

    Args:
    production_df (DataFrame): A DataFrame with production data.
    intensity_df (DataFrame): A DataFrame with intensity data.

    Returns:
    DataFrame: A DataFrame with calculated point-in-time production.
    """
    pit_df = production_df.copy()
    for col in pit_df.columns[9:-6]:  # Iterate over the specific columns
        pit_df[col] = production_df[col] / intensity_df[col]
    return pit_df

# Directories
#Inputs
harvest_dir = "../../data/raw/SPAM/spam2010v2r0_global_harv_area.csv"
physical_dir = "../../data/raw/SPAM/spam2010v2r0_global_phys_area.csv"
production_dir = "../../data/raw/SPAM/spam2010v2r0_global_prod.csv"

#Output
intensity_dir = "../../data/processed/SPAM/"

# Filenames
file_types = ["TI", "TH", "TL", "TS"]

for file_type in file_types:
    harvest_file = f"spam2010V2r0_global_H_{file_type}.csv"
    physical_file = f"spam2010V2r0_global_A_{file_type}.csv"

    # Read files in chunks to manage memory
    harvest_chunks = pd.read_csv(os.path.join(harvest_dir, harvest_file), chunksize=10000)
    physical_chunks = pd.read_csv(os.path.join(physical_dir, physical_file), chunksize=10000)

    for harvest_chunk, physical_chunk in zip(harvest_chunks, physical_chunks):
        # Calculate intensity and append to result file
        result_chunk = calculate_harvest_intensity(harvest_chunk, physical_chunk)
        result_file = f"../../data/processed/SPAM/harvest_intensity_{file_type}.csv"
        if os.path.exists(result_file):
            result_chunk.to_csv(result_file, mode='a', header=False, index=False)
        else:
            result_chunk.to_csv(result_file, index=False)

for file_type in file_types:
    production_file = f"spam2010V2r0_global_P_{file_type}.csv"
    intensity_file = f"harvest_intensity_{file_type}.csv"

    # Read files in chunks to manage memory
    production_chunks = pd.read_csv(os.path.join(production_dir, production_file), chunksize=10000)
    intensity_chunks = pd.read_csv(os.path.join(intensity_dir, intensity_file), chunksize=10000)

    for production_chunk, intensity_chunk in zip(production_chunks, intensity_chunks):
        # Calculate production and append to result file
        result_chunk = calculate_pit_production(production_chunk, intensity_chunk)
        result_file = f"../../data/processed/SPAM/production_pit_{file_type}.csv"
        if os.path.exists(result_file):
            result_chunk.to_csv(result_file, mode='a', header=False, index=False)
        else:
            result_chunk.to_csv(result_file, index=False)


In [5]:
#Cumulative Sum Calculation

# File paths
files = [
    '../../data/processed/SPAM/production_pit_TI.csv',
    '../../data/processed/SPAM/production_pit_TH.csv',
    '../../data/processed/SPAM/production_pit_TS.csv',
    '../../data/processed/SPAM/production_pit_TL.csv'
]


cumulative_sums = {}

for file_path in files:
    # Read each file and calculate column sums
    df = pd.read_csv(file_path)
    column_sums = df.iloc[:, 9:-6].sum(axis=0, skipna=True)

    # Update cumulative sums for each base column
    for column, sum_value in column_sums.items():
        base_column = column.split('_')[0]  # Get the base column name without suffix
        cumulative_sums[base_column] = cumulative_sums.get(base_column, 0) + sum_value


cumulative_sums_df = pd.DataFrame(list(cumulative_sums.items()), columns=['Crop', 'CumulativeSum'])


print(cumulative_sums_df)




    Crop  CumulativeSum
0   whea   5.930520e+08
1   rice   4.725005e+08
2   maiz   7.995436e+08
3   barl   1.347006e+08
4   pmil   2.154770e+07
5   smil   4.637496e+06
6   sorg   5.572797e+07
7   ocer   6.077294e+07
8   pota   3.433514e+08
9   swpo   1.013284e+08
10  yams   4.812227e+07
11  cass   2.336882e+08
12  orts   1.745912e+07
13  bean   2.141557e+07
14  chic   1.033782e+07
15  cowp   4.986169e+06
16  pige   3.805678e+06
17  lent   4.308398e+06
18  opul   1.961451e+07
19  soyb   2.410640e+08
20  grou   3.995570e+07
21  cnut   5.921487e+07
22  oilp   2.282476e+08
23  sunf   3.480039e+07
24  rape   6.210040e+07
25  sesa   4.271791e+06
26  ooil   2.625740e+07
27  sugc   1.711353e+09
28  sugb   2.453048e+08
29  cott   6.763930e+07
30  ofib   4.956373e+06
31  acof   4.430368e+06
32  rcof   3.745078e+06
33  coco   4.384356e+06
34  teas   4.552220e+06
35  toba   7.157627e+06
36  bana   1.047955e+08
37  plnt   2.740164e+07
38  trof   3.625372e+08
39  temf   2.464040e+08
40  vege   8.963

In [6]:
# Save DataFrame to a new CSV file

cumulative_sums_df.to_csv('../../data/processed/pit_production_sum.csv', index=False)